# Carry-On Shopping Answer Analysis  
## Comparison 01, baseline-based multi-prompt version — FIXED  
### 2023/2024 Original Baseline vs One-Product 2026 Current Replacement

This notebook analyzes:

**2023/2024 original baseline vs 2026 current one-product replacement**

Purpose: check how much the actual product page changed over time.  
Interpretation: general website/text change.

## Fixed in this version

This version is stricter about multi-prompt matching. It will not silently summarize only Q1–5 if your current files contain Q6–20 but the matching all-original baseline answer files are missing or not detected.

Main fixes:

- Searches baseline files with `*.txt*`, not only `*.txt`.
- Accepts `2023vs2026`, `2023_2024vs2026`, and similar current-file naming variants.
- Handles product filename typos such as `desley` and `inusea`.
- Ignores GEO files in Comparison 01 by default.
- Matches by `product + question_id`.
- Adds coverage diagnostics for baseline question IDs, current question IDs, matched IDs, and missing baseline IDs.
- Raises a clear error when current questions are missing matching baseline questions, unless you set `ALLOW_PARTIAL_MATCH = True`.

Important: to get Q1–20 results, the baseline folder must contain all-original baseline answer files covering Q1–20. For example, baseline files may be named `baseline_1.txt`, `baseline_6.txt`, `baseline_11.txt`, and `baseline_16.txt`, or `baseline_1.text`, `baseline_6.text`, `baseline_11.text`, and `baseline_16.text`, or any other names, as long as they contain the correct `Question 1:` ... `Question 20:` blocks.


Additional update in this version: baseline discovery accepts both `.txt` and `.text` files, including names such as `baseline_1.text`, `baseline_6.text`, `baseline_11.text`, and `baseline_16.text`.

## Expected folder structure

The notebook searches flexibly, but the main expected structure is:

```text
code/
└── data/
    └── results/
        └── shopping_answers/
            ├── baseline/
            │   ├── baseline.txt
            │   ├── baseline_1.txt or baseline_1.text        # optional
            │   ├── baseline_6.txt or baseline_6.text        # optional
            │   ├── baseline_11.txt or baseline_11.text      # optional
            │   └── baseline_16.txt or baseline_16.text      # optional
            └── 01_2023_2024_original_vs_2026_current/
                ├── beis_2023vs2026_1.txt
                ├── beis_2023vs2026_6.txt
                ├── beis_2023vs2026_16.txt
                ├── delsey_2023vs2026_1.txt
                ├── delsey_2023vs2026_6.txt
                ├── desley_2023vs2026_16.txt
                └── ...
```

Optional product-page text folders used for PAIS-style text similarity:

```text
code/data/filtered/old/
code/data/filtered/current/
```


## CSV outputs

The notebook saves CSV files to:

```text
code/data/results/tables/01_2023_2024_original_vs_2026_current_baseline_based/
```

Saved files:

```text
file_manifest_comparison01.csv
baseline_parsed_answers.csv
current_parsed_answers_by_file.csv
baseline_question_product_metrics.csv
current_question_product_metrics.csv
matched_current_baseline_question_product_delta.csv
baseline_product_summary_matched.csv
current_product_summary_matched.csv
current_vs_baseline_delta_by_product.csv
overall_current_vs_baseline_summary.csv
product_text_load_status.csv
```


In [1]:
from pathlib import Path
import re
import json
import math
import numpy as np
import pandas as pd

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_rows", 300)


In [2]:
# =============================
# Configuration
# =============================

PRODUCTS = ["Monos", "BEIS", "Travelpro", "Delsey", "INUSA"]
N_PRODUCTS = len(PRODUCTS)

COMPARISON_NAME = "01_2023_2024_original_vs_2026_current"
OUTPUT_NAME = "01_2023_2024_original_vs_2026_current_baseline_based"

# Optional manual overrides.
# If automatic path detection fails, set these manually.
BASELINE_DIR = None
BASELINE_FILES = None     # Example: [Path(".../baseline_1.txt"), Path(".../baseline_6.text")]
COMPARISON_DIR = None

# Optional manual overrides for product-page text folders.
# These are used only for PAIS-style product-text-to-answer similarity.
OLD_TEXT_DIR = None
CURRENT_TEXT_DIR = None

# For Comparison 01, GEO files should not be analyzed even if they are in the same folder.
# If your Q11-15 files are actually current-vs-2026 runs but accidentally contain "GEO" in the filename,
# rename them to something like beis_2023vs2026_11.txt instead of changing this flag.
IGNORE_GEO_FILES_IN_COMPARISON01 = True

# IMPORTANT FIX:
# If False, the notebook raises an error when some current question IDs do not have matching baseline question IDs.
# This prevents accidentally reporting only Q1-5 when Q6-20 current files exist but baseline files are missing.
ALLOW_PARTIAL_MATCH = False

# Expected final benchmark questions. Used for diagnostics only.
EXPECTED_QUESTION_IDS = list(range(1, 21))

# If you want to restrict analysis to specific question IDs, set a list like:
# QUESTION_ID_FILTER = list(range(1, 21))
QUESTION_ID_FILTER = None


In [3]:
# =============================
# Robust path detection
# =============================

def get_candidate_roots():
    cwd = Path.cwd().resolve()
    roots = []
    for p in [
        cwd,
        cwd.parent,
        cwd.parent.parent,
        cwd.parent.parent.parent,
        cwd / "code",
        cwd.parent / "code",
        cwd.parent.parent / "code",
        cwd.parent.parent.parent / "code",
    ]:
        try:
            rp = p.resolve()
        except Exception:
            rp = p
        if p.exists() and rp not in roots:
            roots.append(rp)
    return roots

def find_dir(rel_candidates):
    for root in get_candidate_roots():
        for rel in rel_candidates:
            candidate = root / rel
            if candidate.exists() and candidate.is_dir():
                return candidate.resolve()
    return None

def normalize_filename(name: str) -> str:
    return name.lower().replace("é", "e").replace(" ", "").replace("-", "_")

PRODUCT_ALIASES = {
    "BEIS": ["beis", "béis"],
    "Delsey": ["delsey", "desley", "delesy"],
    "INUSA": ["inusa", "inusea"],
    "Monos": ["monos"],
    "Travelpro": ["travelpro", "travel"],
}

def infer_product_from_filename(filename: str) -> str:
    name = normalize_filename(filename)
    for product, aliases in PRODUCT_ALIASES.items():
        if any(alias in name for alias in aliases):
            return product
    return "Unknown"

def is_geo_filename(filename: str) -> bool:
    name = normalize_filename(filename)
    compact = re.sub(r"[^a-z0-9]", "", name)
    return "geo" in compact or "rewrite" in compact

def is_text_like(path: Path) -> bool:
    """Accept normal text answer files.

    Supports both .txt and .text because some prompt outputs may be saved as
    baseline_1.text, baseline_6.text, etc. Also accepts double extensions that
    Windows/Notepad sometimes creates, such as .txt.txt or .text.txt.
    """
    n = path.name.lower()
    return (
        n.endswith(".txt")
        or n.endswith(".txt.txt")
        or n.endswith(".text")
        or n.endswith(".text.txt")
    )

def is_current_replacement_filename(filename: str) -> bool:
    # Accept 2023vs2026 / 2023_2024vs2026 variants, but reject GEO files by default.
    name = normalize_filename(filename)
    if filename.startswith("."):
        return False
    if not is_text_like(Path(filename)):
        return False
    if IGNORE_GEO_FILES_IN_COMPARISON01 and is_geo_filename(name):
        return False
    if infer_product_from_filename(name) == "Unknown":
        return False
    compact = re.sub(r"[^a-z0-9]", "", name)
    # Examples accepted: beis_2023vs2026_1.txt, beis_2023_2024vs2026_1.txt, beis_oldvs2026_16.txt
    if "vs2026" in compact or "2026" in compact:
        return True
    return False

def infer_prompt_set_from_question_ids(question_ids):
    if not question_ids:
        return np.nan, np.nan, ""
    qmin, qmax = int(min(question_ids)), int(max(question_ids))
    return qmin, qmax, f"{qmin}-{qmax}"

def sort_prompt_set_labels(labels):
    """Sort labels like 1-5, 6-10, 11-15, 16-20 in numeric order."""
    clean = [str(x) for x in labels if str(x).strip() and str(x).lower() != "nan"]
    def key(label):
        m = re.search(r"(\d+)", label)
        return int(m.group(1)) if m else 10**9
    return sorted(set(clean), key=key)

def join_prompt_set_labels(labels):
    return ";".join(sort_prompt_set_labels(labels))

def discover_baseline_files(baseline_dir: Path):
    """Find all all-original baseline answer files.

    Uses both rglob('*.txt*') and rglob('*.text*') so files like baseline_16.txt.txt or baseline_16.text are included.
    Excludes obvious comparison / GEO / current replacement files.
    """
    if baseline_dir is None:
        return []
    # Accept both .txt and .text baseline files.
    # Examples: baseline.txt, baseline_1.txt, baseline_6.text, baseline_16.text.txt.
    candidates = []
    for pattern in ["*.txt*", "*.text*"]:
        candidates.extend([p for p in baseline_dir.rglob(pattern) if p.is_file() and is_text_like(p)])
    files = sorted(set(candidates))
    valid = []
    for p in files:
        n = normalize_filename(p.name)
        compact = re.sub(r"[^a-z0-9]", "", n)
        # Exclude counterfactual/comparison outputs if accidentally copied into the baseline folder.
        banned_tokens = ["vs2026", "vsgeo", "2023vs2026", "20232024vs2026", "20232024vsgeo", "geo_rewrite", "georewrite"]
        if any(tok in compact for tok in banned_tokens):
            continue
        valid.append(p)
    return valid

if BASELINE_DIR is None:
    BASELINE_DIR = find_dir([
        Path("data/results/shopping_answers/baseline"),
        Path("code/data/results/shopping_answers/baseline"),
        Path("results/shopping_answers/baseline"),
        Path("data/results/shopping_answers/00_baseline"),
        Path("code/data/results/shopping_answers/00_baseline"),
        Path("results/shopping_answers/00_baseline"),
        Path("data/results/shopping_answers/00_all_original_2023_2024"),
        Path("code/data/results/shopping_answers/00_all_original_2023_2024"),
        Path("results/shopping_answers/00_all_original_2023_2024"),
    ])
else:
    BASELINE_DIR = Path(BASELINE_DIR).resolve()

if COMPARISON_DIR is None:
    COMPARISON_DIR = find_dir([
        Path("data/results/shopping_answers") / COMPARISON_NAME,
        Path("code/data/results/shopping_answers") / COMPARISON_NAME,
        Path("results/shopping_answers") / COMPARISON_NAME,
    ])
else:
    COMPARISON_DIR = Path(COMPARISON_DIR).resolve()

if OLD_TEXT_DIR is None:
    OLD_TEXT_DIR = find_dir([
        Path("data/filtered/old"),
        Path("code/data/filtered/old"),
        Path("data/results/filtered/old"),
        Path("code/data/results/filtered/old"),
    ])
else:
    OLD_TEXT_DIR = Path(OLD_TEXT_DIR).resolve()

if CURRENT_TEXT_DIR is None:
    CURRENT_TEXT_DIR = find_dir([
        Path("data/filtered/current"),
        Path("code/data/filtered/current"),
        Path("data/results/filtered/current"),
        Path("code/data/results/filtered/current"),
    ])
else:
    CURRENT_TEXT_DIR = Path(CURRENT_TEXT_DIR).resolve()

if BASELINE_FILES is None:
    BASELINE_FILES = discover_baseline_files(BASELINE_DIR)
else:
    BASELINE_FILES = [Path(p).resolve() for p in BASELINE_FILES]

if COMPARISON_DIR is not None:
    candidates = []
    for pattern in ["*.txt*", "*.text*"]:
        candidates.extend([p for p in COMPARISON_DIR.glob(pattern) if p.is_file() and is_text_like(p)])
    all_comparison_txt_files = sorted(set(candidates))
else:
    all_comparison_txt_files = []

TXT_FILES = [p for p in all_comparison_txt_files if is_current_replacement_filename(p.name)]
IGNORED_TXT_FILES = [p for p in all_comparison_txt_files if p not in TXT_FILES]

print("Current working directory:", Path.cwd().resolve())
print("Detected baseline directory:", BASELINE_DIR)
print("Detected baseline files:")
for p in BASELINE_FILES:
    print("  -", p.name)
print("Detected comparison directory:", COMPARISON_DIR)
print("Detected old text directory:", OLD_TEXT_DIR)
print("Detected current text directory:", CURRENT_TEXT_DIR)

if not BASELINE_FILES:
    raise FileNotFoundError(
        "Could not find baseline .txt/.text files. "
        "Please set BASELINE_DIR or BASELINE_FILES manually in the configuration cell. "
        "For Q1-20 analysis, baseline files must cover Q1-20."
    )

if COMPARISON_DIR is None:
    raise FileNotFoundError(
        "Could not find the comparison folder. "
        "Please set COMPARISON_DIR manually in the configuration cell."
    )

print(f"\nIncluded {len(TXT_FILES)} one-product-current txt files:")
for p in TXT_FILES:
    print("  -", p.name, "=>", infer_product_from_filename(p.name))

print(f"\nIgnored {len(IGNORED_TXT_FILES)} txt-like files:")
for p in IGNORED_TXT_FILES:
    reason = "GEO file" if is_geo_filename(p.name) else "not recognized as current replacement"
    print("  -", p.name, f"({reason})")

if not TXT_FILES:
    raise FileNotFoundError(f"No current-replacement .txt/.text files found in {COMPARISON_DIR}")


Current working directory: C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\code_V2
Detected baseline directory: C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\baseline
Detected baseline files:
  - baseline_1.txt
  - baseline_11.txt
  - baseline_16.txt
  - baseline_6.txt
Detected comparison directory: C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current
Detected old text directory: C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\data\filtered\old
Detected current text directory: C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\data\filtered\current

Included 20 one-product-current txt files:
  - beis_2023vs2026_1.txt => BEIS
  - beis_2023vs2026_11.txt => BEIS
  - beis_2023vs2026_16.txt => BEIS
  - beis_2023vs2026_6.txt => BEIS
  - delsey_2023vs2026_1.txt => Delsey
  - delsey_2023vs2026_11.txt => Delsey
  - delsey_2023vs2026_6.

In [4]:
# =============================
# Output directory
# =============================

# Based on your structure, if COMPARISON_DIR is:
# code/data/results/shopping_answers/01_...
# then tables go to:
# code/data/results/tables/01_..._baseline_based
RESULTS_DIR = COMPARISON_DIR.parents[1]  # .../results
OUTPUT_DIR = RESULTS_DIR / "tables" / OUTPUT_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

file_manifest_rows = []
for p in BASELINE_FILES:
    file_manifest_rows.append({
        "file_role": "baseline_included",
        "path": str(p),
        "filename": p.name,
        "detected_product": "",
        "included": True,
        "reason": "baseline answer file",
    })
for p in TXT_FILES:
    file_manifest_rows.append({
        "file_role": "comparison01_current_included",
        "path": str(p),
        "filename": p.name,
        "detected_product": infer_product_from_filename(p.name),
        "included": True,
        "reason": "current replacement file",
    })
for p in IGNORED_TXT_FILES:
    file_manifest_rows.append({
        "file_role": "comparison01_ignored",
        "path": str(p),
        "filename": p.name,
        "detected_product": infer_product_from_filename(p.name),
        "included": False,
        "reason": "GEO file ignored" if is_geo_filename(p.name) else "not recognized as current replacement",
    })

file_manifest_comparison01 = pd.DataFrame(file_manifest_rows)

print("Results directory:", RESULTS_DIR)
print("CSV output directory:", OUTPUT_DIR)
display(file_manifest_comparison01)


Results directory: C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results
CSV output directory: C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\tables\01_2023_2024_original_vs_2026_current_baseline_based


,file_role,path,filename,detected_product,included,reason
0,baseline_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\baseline\baseline_1.txt,baseline_1.txt,,True,baseline answer file
1,baseline_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\baseline\baseline_11.txt,baseline_11.txt,,True,baseline answer file
2,baseline_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\baseline\baseline_16.txt,baseline_16.txt,,True,baseline answer file
3,baseline_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\baseline\baseline_6.txt,baseline_6.txt,,True,baseline answer file
4,comparison01_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\beis_2023vs2026_1.txt,beis_2023vs2026_1.txt,BEIS,True,current replacement file
5,comparison01_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\beis_2023vs2026_11.txt,beis_2023vs2026_11.txt,BEIS,True,current replacement file
6,comparison01_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\beis_2023vs2026_16.txt,beis_2023vs2026_16.txt,BEIS,True,current replacement file
7,comparison01_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\beis_2023vs2026_6.txt,beis_2023vs2026_6.txt,BEIS,True,current replacement file
8,comparison01_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\delsey_2023vs2026_1.txt,delsey_2023vs2026_1.txt,Delsey,True,current replacement file
9,comparison01_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\delsey_2023vs2026_11.txt,delsey_2023vs2026_11.txt,Delsey,True,current replacement file


## Parse answer files

The parser expects each answer file to contain question blocks such as:

```text
Question 1:
Answer:
...

Question 2:
Answer:
...

Overall shopper takeaway:
...
```

This version accepts any question ID range, so it can parse prompt sets 1–5, 6–10, 11–15, and 16–20.


In [5]:
# =============================
# Parsing functions
# =============================

QUESTION_BLOCK_RE = re.compile(
    r"Question\s*(\d+)\s*:\s*(.*?)(?=\n\s*Question\s*\d+\s*:|\n\s*Overall shopper takeaway\s*:|\Z)",
    flags=re.IGNORECASE | re.DOTALL,
)

OVERALL_RE = re.compile(
    r"Overall shopper takeaway\s*:\s*(.*)\Z",
    flags=re.IGNORECASE | re.DOTALL,
)

def clean_answer_block(block: str) -> str:
    block = block.strip()
    block = re.sub(r"^\s*Answer\s*:\s*", "", block, flags=re.IGNORECASE).strip()
    block = re.sub(r"\n{3,}", "\n\n", block)
    return block

def parse_answer_file(path: Path, run_type: str, focal_current_product: str = None) -> pd.DataFrame:
    text = path.read_text(encoding="utf-8-sig", errors="replace").replace("\r\n", "\n").replace("\r", "\n")
    rows = []
    qids_found = []

    for m in QUESTION_BLOCK_RE.finditer(text):
        qid = int(m.group(1))
        if QUESTION_ID_FILTER is not None and qid not in set(QUESTION_ID_FILTER):
            continue
        answer = clean_answer_block(m.group(2))
        qids_found.append(qid)
        rows.append({
            "source_file": path.name,
            "source_path": str(path),
            "run_type": run_type,
            "focal_current_product": focal_current_product,
            "comparison": COMPARISON_NAME,
            "question_id": qid,
            "answer_text": answer,
        })

    overall = ""
    om = OVERALL_RE.search(text)
    if om:
        overall = om.group(1).strip()

    qmin, qmax, qlabel = infer_prompt_set_from_question_ids(qids_found)

    df = pd.DataFrame(rows)
    if not df.empty:
        df["prompt_set_start"] = qmin
        df["prompt_set_end"] = qmax
        df["prompt_set_label"] = qlabel
        df["overall_shopper_takeaway"] = overall
    return df

baseline_parts = [
    parse_answer_file(p, run_type="baseline_all_original", focal_current_product=None)
    for p in BASELINE_FILES
]
baseline_parts = [df for df in baseline_parts if not df.empty]
baseline_answers = pd.concat(baseline_parts, ignore_index=True) if baseline_parts else pd.DataFrame()

current_parts = [
    parse_answer_file(
        p,
        run_type="one_product_current",
        focal_current_product=infer_product_from_filename(p.name),
    )
    for p in TXT_FILES
]
current_parts = [df for df in current_parts if not df.empty]
current_answers = pd.concat(current_parts, ignore_index=True) if current_parts else pd.DataFrame()

if baseline_answers.empty:
    raise ValueError("No question blocks were parsed from baseline files.")

if current_answers.empty:
    raise ValueError("No question blocks were parsed from current comparison files.")

# Remove exact duplicate parsed rows if the same file was accidentally included twice.
dedup_cols = ["source_file", "run_type", "focal_current_product", "question_id", "answer_text"]
baseline_answers = baseline_answers.drop_duplicates(subset=dedup_cols).reset_index(drop=True)
current_answers = current_answers.drop_duplicates(subset=dedup_cols).reset_index(drop=True)

baseline_qids = sorted(map(int, baseline_answers["question_id"].unique()))
current_qids = sorted(map(int, current_answers["question_id"].unique()))

print("Baseline parsed rows:", len(baseline_answers))
display(baseline_answers[["source_file", "prompt_set_label", "question_id", "answer_text"]].head(30))

print("Current counterfactual parsed rows:", len(current_answers))
display(current_answers[["source_file", "focal_current_product", "prompt_set_label", "question_id", "answer_text"]].head(30))

print("Focal current products detected by file:")
display(
    current_answers[["source_file", "focal_current_product", "prompt_set_label"]]
    .drop_duplicates()
    .sort_values(["focal_current_product", "prompt_set_label", "source_file"])
)

coverage_by_file = pd.concat([
    baseline_answers.groupby(["run_type", "source_file"], as_index=False).agg(
        focal_current_product=("focal_current_product", lambda x: ""),
        question_ids=("question_id", lambda x: ",".join(map(str, sorted(set(map(int, x)))))),
        n_questions=("question_id", "nunique"),
        prompt_set_label=("prompt_set_label", lambda x: ";".join(sorted(set(map(str, x)))))
    ),
    current_answers.groupby(["run_type", "source_file", "focal_current_product"], as_index=False).agg(
        question_ids=("question_id", lambda x: ",".join(map(str, sorted(set(map(int, x)))))),
        n_questions=("question_id", "nunique"),
        prompt_set_label=("prompt_set_label", lambda x: ";".join(sorted(set(map(str, x)))))
    )
], ignore_index=True, sort=False)

print("Question coverage by parsed file:")
display(coverage_by_file.sort_values(["run_type", "focal_current_product", "source_file"]))

question_coverage_summary = pd.DataFrame([{
    "baseline_question_ids": ",".join(map(str, baseline_qids)),
    "current_question_ids": ",".join(map(str, current_qids)),
    "expected_question_ids": ",".join(map(str, EXPECTED_QUESTION_IDS)),
    "missing_from_baseline_vs_expected": ",".join(map(str, sorted(set(EXPECTED_QUESTION_IDS) - set(baseline_qids)))),
    "missing_from_current_vs_expected": ",".join(map(str, sorted(set(EXPECTED_QUESTION_IDS) - set(current_qids)))),
}])

print("Overall question coverage summary:")
display(question_coverage_summary)


Baseline parsed rows: 20


,source_file,prompt_set_label,question_id,answer_text
0,baseline_1.txt,1-5,1,"For a 3-5 day trip, I would recommend the Delsey CHATELET AIR 2.0 Carry-On Plus Spinner first because the page specifically says the Carry-On Plus is ideal for 3-5 day trips, f..."
1,baseline_1.txt,1-5,2,"The Travelpro seems best if durability and protecting belongings matter most because its page describes an ultra-strong, easy-to-clean 100% polycarbonate shell that flexes on i..."
2,baseline_1.txt,1-5,3,"The Travelpro appears easiest to move through airports, streets, and crowded spaces because its PrecisionGlide system is described as delivering precise control and effortless ..."
3,baseline_1.txt,1-5,4,"BEIS seems best for packing organization because it includes a butterfly opening, a U-zip flap with one zip pocket and one frosted PVC zip pocket, a detachable compression flap..."
4,baseline_1.txt,1-5,5,"For the best overall purchase confidence, I would choose Travelpro because it gives exact overall dimensions, a 45 L volume, sizer-bin testing for overhead bins on most major d..."
5,baseline_11.txt,11-15,11,"For stricter carry-on rules, Monos sounds safest from the airline-fit wording because its page says the Carry-On is designed to fit in the overhead bin of almost any flight, an..."
6,baseline_11.txt,11-15,12,"Travelpro gives the clearest easy-clean and appearance-protection information: it says the 100% polycarbonate shell is easy to clean, flexes upon impact to prevent cracking, an..."
7,baseline_11.txt,11-15,13,"Delsey provides the clearest sustainability-related claim because its feature list says the bag is made with recycled materials, while also listing durability features such as ..."
8,baseline_11.txt,11-15,14,"Travelpro seems most useful for travel-day convenience overall because it combines external USB-A and USB-C fast-charge ports, a dedicated external-access battery pocket with F..."
9,baseline_11.txt,11-15,15,"Travelpro has the strongest information quality because it provides overall and case dimensions, weight, volume, sizer-bin testing, an explicit warning that the fully expanded ..."


Current counterfactual parsed rows: 100


,source_file,focal_current_product,prompt_set_label,question_id,answer_text
0,beis_2023vs2026_1.txt,BEIS,1-5,1,"For a 3-5 day trip, I would recommend the Delsey CHATELET AIR 2.0 Carry-On Plus Spinner as the most directly matched option because its page specifically says the Carry-On Plus..."
1,beis_2023vs2026_1.txt,BEIS,1-5,2,"If durability and protecting belongings are the top priorities, I would recommend Travelpro first, with Delsey as a close second. Travelpro uses an ultra-strong, easy-to-clean ..."
2,beis_2023vs2026_1.txt,BEIS,1-5,3,Travelpro seems easiest to move through airports and crowded spaces because its page gives the most specific mobility system: PrecisionGlide® for precise control and effortless...
3,beis_2023vs2026_1.txt,BEIS,1-5,4,"For packing organization, I would recommend BÉIS because it has the most detailed organization setup in the provided pages. Its interior includes a butterfly opening, easy-to-c..."
4,beis_2023vs2026_1.txt,BEIS,1-5,5,"For the best overall purchase confidence, I would choose Travelpro if the shopper is mainly flying on major domestic airlines. Its page provides exact overall dimensions, case ..."
5,beis_2023vs2026_11.txt,BEIS,11-15,11,"For stricter carry-on rules, the safest-looking option by airline-fit language is Monos because the page says the Carry-On is “designed to fit in the overhead bin of almost any..."
6,beis_2023vs2026_11.txt,BEIS,11-15,12,"Travelpro gives the strongest clean-and-good-looking evidence because it says the 100% polycarbonate shell is easy to clean, the textured finish helps reduce the visibility of ..."
7,beis_2023vs2026_11.txt,BEIS,11-15,13,"Delsey provides the strongest sustainability-related evidence because its feature list says “Made with Recycled Materials,” and the same page also supports durability with 100%..."
8,beis_2023vs2026_11.txt,BEIS,11-15,14,"BEIS seems the most broadly useful for travel-day convenience because it combines a built-in weight indicator, a retractable bag strap that can hold up to 15 lbs, 2 inches of e..."
9,beis_2023vs2026_11.txt,BEIS,11-15,15,"Travelpro has the strongest information quality overall because it gives overall and case dimensions, weight, volume, material, sizer-bin language, an expansion caveat, cleanin..."


Focal current products detected by file:


,source_file,focal_current_product,prompt_set_label
0,beis_2023vs2026_1.txt,BEIS,1-5
5,beis_2023vs2026_11.txt,BEIS,11-15
10,beis_2023vs2026_16.txt,BEIS,16-20
15,beis_2023vs2026_6.txt,BEIS,6-10
20,delsey_2023vs2026_1.txt,Delsey,1-5
25,delsey_2023vs2026_11.txt,Delsey,11-15
35,desley_2023vs2026_16.txt,Delsey,16-20
30,delsey_2023vs2026_6.txt,Delsey,6-10
40,inusa_2023vs2026_1.txt,INUSA,1-5
45,inusa_2023vs2026_11.txt,INUSA,11-15


Question coverage by parsed file:


,run_type,source_file,focal_current_product,question_ids,n_questions,prompt_set_label
0,baseline_all_original,baseline_1.txt,,"1,2,3,4,5",5,1-5
1,baseline_all_original,baseline_11.txt,,"11,12,13,14,15",5,11-15
2,baseline_all_original,baseline_16.txt,,"16,17,18,19,20",5,16-20
3,baseline_all_original,baseline_6.txt,,"6,7,8,9,10",5,6-10
4,one_product_current,beis_2023vs2026_1.txt,BEIS,"1,2,3,4,5",5,1-5
5,one_product_current,beis_2023vs2026_11.txt,BEIS,"11,12,13,14,15",5,11-15
6,one_product_current,beis_2023vs2026_16.txt,BEIS,"16,17,18,19,20",5,16-20
7,one_product_current,beis_2023vs2026_6.txt,BEIS,"6,7,8,9,10",5,6-10
8,one_product_current,delsey_2023vs2026_1.txt,Delsey,"1,2,3,4,5",5,1-5
9,one_product_current,delsey_2023vs2026_11.txt,Delsey,"11,12,13,14,15",5,11-15


Overall question coverage summary:


,baseline_question_ids,current_question_ids,expected_question_ids,missing_from_baseline_vs_expected,missing_from_current_vs_expected
0,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20","1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20","1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",,


### Baseline file naming note

For Q1–20, make sure the baseline folder contains all-original answers covering the same question IDs as the current replacement files. The notebook now accepts either `.txt` or `.text` baseline files, for example:

```text
baseline_1.text    # Q1–5
baseline_6.text    # Q6–10
baseline_11.text   # Q11–15
baseline_16.text   # Q16–20
```

The exact file names do not matter as long as the files contain `Question 1:`, `Question 6:`, `Question 11:`, and `Question 16:` blocks respectively.

## Load product-page texts

The PAIS-style score uses product-page text similarity to cited answer sentences.

If the text files are not found, the notebook still computes citation/rank/feature metrics, while the TF-IDF and n-gram parts of PAIS are set to zero.


In [6]:
# =============================
# Product text loading
# =============================

PRODUCT_FILE_STEMS = {
    "BEIS": "beis_carry_on_roller_filtered.txt",
    "Delsey": "delsey_chatelet_air_carry_on_plus_filtered.txt",
    "INUSA": "inusa_ally_carry_on_filtered.txt",
    "Monos": "monos_carry_on_filtered.txt",
    "Travelpro": "travelpro_platinum_elite_carry_on_filtered.txt",
}

def read_text_if_exists(path):
    if path is not None and path.exists():
        return path.read_text(encoding="utf-8-sig", errors="replace")
    return ""

product_texts = {}
load_rows = []

for version_status, folder in [
    ("original_2023_2024", OLD_TEXT_DIR),
    ("current_2026", CURRENT_TEXT_DIR),
]:
    for product, filename in PRODUCT_FILE_STEMS.items():
        path = folder / filename if folder is not None else None
        text = read_text_if_exists(path)
        product_texts[(version_status, product)] = text
        load_rows.append({
            "version_status": version_status,
            "product": product,
            "path": str(path) if path is not None else "",
            "loaded": bool(text.strip()),
            "characters": len(text),
        })

product_text_load_status = pd.DataFrame(load_rows)
display(product_text_load_status)


,version_status,product,path,loaded,characters
0,original_2023_2024,BEIS,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\data\filtered\old\beis_carry_on_roller_filtered.txt,True,2375
1,original_2023_2024,Delsey,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\data\filtered\old\delsey_chatelet_air_carry_on_plus_filtered.txt,True,3572
2,original_2023_2024,INUSA,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\data\filtered\old\inusa_ally_carry_on_filtered.txt,True,1572
3,original_2023_2024,Monos,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\data\filtered\old\monos_carry_on_filtered.txt,True,1898
4,original_2023_2024,Travelpro,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\data\filtered\old\travelpro_platinum_elite_carry_on_filtered.txt,True,6179
5,current_2026,BEIS,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\data\filtered\current\beis_carry_on_roller_filtered.txt,True,2312
6,current_2026,Delsey,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\data\filtered\current\delsey_chatelet_air_carry_on_plus_filtered.txt,True,2124
7,current_2026,INUSA,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\data\filtered\current\inusa_ally_carry_on_filtered.txt,True,2128
8,current_2026,Monos,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\data\filtered\current\monos_carry_on_filtered.txt,True,3797
9,current_2026,Travelpro,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\data\filtered\current\travelpro_platinum_elite_carry_on_filtered.txt,True,5621


## Citation, rank, feature, and PAIS-style metrics


In [7]:
# =============================
# Citation and feature extraction
# =============================

PRODUCT_PATTERN = re.compile(r"\[(Monos|BÉIS|BEIS|Travelpro|Delsey|INUSA)\]", flags=re.IGNORECASE)

CANONICAL = {
    "monos": "Monos",
    "beis": "BEIS",
    "béis": "BEIS",
    "travelpro": "Travelpro",
    "delsey": "Delsey",
    "inusa": "INUSA",
}

# Feature categories are intentionally broad and product-page-oriented.
# The added categories help later prompt sets 11-20.
FEATURE_PATTERNS = {
    "trip_length": r"\b(3-5|3–5|4-6|4–6|2-5|2–5|day trip|trip length|overnight|weekend|getaway|outfits?)\b",
    "overhead_bin_or_airline_fit": r"\b(overhead|carry-on|carry on|sizer|airline|domestic airlines?|flight|bin|gate-check|gate check|restrictions?)\b",
    "dimensions": r"\b(dimensions?|measurements?|inches|inch|in\.|cm|height|width|depth|linear|exterior|interior)\b",
    "weight": r"\b(weight|lbs?|pounds?|kg|lightweight|lighter|lift)\b",
    "capacity_volume": r"\b(capacity|volume|liters?|litres?|L\b|packing space|room|extra room|capacity-to-weight)\b",
    "material_shell": r"\b(polycarbonate|PC/ABS|shell|hard shell|hardside|hard-sided|micro diamond|recycled materials?)\b",
    "durability_protection": r"\b(durable|durability|protect|protection|impact|cracking|corner|guard|armor|scuff|scratch|resistant|resilience|shock|rough handling)\b",
    "wheels_mobility": r"\b(wheels?|spinner|rolling|roll|glide|gliding|maneuver|manoeuvre|mobility|PrecisionGlide|MagnaTrac|Hinomoto|360|crowded|streets?)\b",
    "handle": r"\b(handle|telescopic|trolley|PowerScope|Contour Grip|grip|cushioned|soft-grip|GEL|ergonomic)\b",
    "lock_security": r"\b(TSA|lock|security|secure|Travel Sentry|combination|SECURITECH|zip|zipper|intrusion)\b",
    "expansion": r"\b(expand|expandable|expansion|2-inch|2 inch|extra space|overpacker|overpackers)\b",
    "packing_organization": r"\b(compartment|compression|pocket|divider|strap|laundry|shoe|mesh|organize|organization|clamshell|butterfly|valuables|pouch|small items?)\b",
    "warranty_trial_return": r"\b(warranty|trial|return|returns|lifetime|10-year|100-day|promise|coverage|reassurance|purchase confidence)\b",
    "price_value": r"\b(price|priced|cost|lower-priced|budget|value|sale|discount|investment)\b",
    "electronics_convenience": r"\b(USB|USB-A|USB-C|charging|charger|battery|power bank|phone|electronics?|device|tracking|tracker|tech-heavy)\b",
    "cleaning_care": r"\b(clean|cleaning|wipe|wipe-clean|stain|water-resistant|water resistant|scratch|scuff|finish|marks?|dust bag|care)\b",
    "sustainability": r"\b(sustainable|sustainability|responsible|recycled|B Corporation|B Corp|eco-friendly|eco|materials?)\b",
    "comparison_ready_evidence": r"\b(specs?|specifications?|details?|feature details?|trust signals?|buyer caveats?|compare|comparison|clear information|information quality)\b",
    "tradeoff_or_limitation": r"\b(limitation|limit|trade-off|tradeoff|may not|does not|caution|check|vary|not provide|less detail|less complete|harder to compare|however|but|weaker choice)\b",
}

def canonical_product(label: str) -> str:
    return CANONICAL.get(label.lower(), label)

def citation_sequence(text: str):
    return [canonical_product(m.group(1)) for m in PRODUCT_PATTERN.finditer(text or "")]

# Provider-style mention metrics. These parse natural brand/product mentions after
# removing controlled bracket citations such as [Monos]. Citation metrics remain
# separate from plain brand-mention metrics.
PRODUCT_MENTION_ALIASES = {
    "Monos": ["Monos"],
    "BEIS": ["BÉIS", "BEIS", "Beis"],
    "Travelpro": ["Travelpro", "Platinum Elite"],
    "Delsey": ["Delsey", "Chatelet", "Châtelet"],
    "INUSA": ["INUSA", "Inusa", "Ally"],
}

MENTION_CANONICAL = {}
for _canon, _aliases in PRODUCT_MENTION_ALIASES.items():
    for _alias in _aliases:
        MENTION_CANONICAL[_alias.lower()] = _canon

# Match longer aliases first so "Platinum Elite" is not split into smaller fragments.
MENTION_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(a) for aliases in PRODUCT_MENTION_ALIASES.values() for a in sorted(aliases, key=len, reverse=True)) + r")\b",
    flags=re.IGNORECASE,
)

POSITIVE_CONTEXT_PATTERN = re.compile(
    r"\b(best|strong|recommend(?:ed)?|useful|durable|smooth|secure|clear|confidence|better|stand(?:s)? out|excellent|ideal|good|reliable|premium|spacious|lightweight|protect(?:s|ive|ion)?|organized|convenient|easy|fits?|warranty|trial|trusted|effortless|precise|room|capacity)\b",
    flags=re.IGNORECASE,
)
NEGATIVE_CONTEXT_PATTERN = re.compile(
    r"\b(caution|limitation|limited|not|may not|does not|doesn't|heavy|larger|weaker|less|risk|gate-check|gate check|check|expensive|unclear|missing|lacks?|trade[- ]?off|downside|fully expanded|may exceed|harder)\b",
    flags=re.IGNORECASE,
)

def answer_without_product_citations(text: str) -> str:
    return PRODUCT_PATTERN.sub(" ", text or "")

def canonical_mention(label: str) -> str:
    return MENTION_CANONICAL.get((label or "").lower(), label)

def brand_mention_sequence(text: str):
    clean_text = answer_without_product_citations(text)
    return [canonical_mention(m.group(1)) for m in MENTION_PATTERN.finditer(clean_text or "")]

def product_mentioned_in_text(text: str, product: str) -> bool:
    return product in set(brand_mention_sequence(text))

def mentioned_sentences_for_product(answer: str, product: str):
    sentences = []
    for sent in split_sentences(answer_without_product_citations(answer)):
        if product_mentioned_in_text(sent, product):
            sentences.append(sent)
    return sentences

def simple_brand_sentiment(context_text: str):
    # Lightweight dictionary proxy for answer-context tone. This is descriptive only,
    # not a substitute for a validated sentiment model.
    pos = len(POSITIVE_CONTEXT_PATTERN.findall(context_text or ""))
    neg = len(NEGATIVE_CONTEXT_PATTERN.findall(context_text or ""))
    denom = pos + neg
    score = (pos - neg) / denom if denom else 0.0
    return float(score), int(pos), int(neg)


def split_sentences(text: str):
    parts = re.split(r"(?<=[.!?])\s+", (text or "").strip())
    return [p.strip() for p in parts if p.strip()]

def cited_sentences_for_product(answer: str, product: str):
    sentences = []
    for sent in split_sentences(answer):
        if product in set(citation_sequence(sent)):
            sentences.append(sent)
    return sentences

def categories_in_text(text: str):
    found = []
    for cat, pat in FEATURE_PATTERNS.items():
        if re.search(pat, text or "", flags=re.IGNORECASE):
            found.append(cat)
    return sorted(set(found))

def product_feature_categories(answer: str, product: str):
    cats = set()
    for sent in cited_sentences_for_product(answer, product):
        cats.update(categories_in_text(sent))
    return sorted(cats)

def word_count(text: str) -> int:
    return len(re.findall(r"\b\w+\b", text or ""))

def simple_ngram_overlap(a: str, b: str, n: int = 2) -> float:
    def toks(x):
        return re.findall(r"\b[a-zA-Z0-9]+\b", (x or "").lower())
    ta, tb = toks(a), toks(b)
    if len(ta) < n or len(tb) < n:
        return 0.0
    nga = set(tuple(ta[i:i+n]) for i in range(len(ta)-n+1))
    ngb = set(tuple(tb[i:i+n]) for i in range(len(tb)-n+1))
    if not nga or not ngb:
        return 0.0
    return len(nga & ngb) / len(ngb)

def tfidf_similarity_to_product_text(product_text: str, cited_answer_text: str) -> float:
    if not SKLEARN_AVAILABLE:
        return 0.0
    if not product_text.strip() or not cited_answer_text.strip():
        return 0.0
    try:
        vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
        X = vec.fit_transform([product_text, cited_answer_text])
        return float(cosine_similarity(X[0:1], X[1:2])[0, 0])
    except Exception:
        return 0.0

def build_question_product_metrics(parsed_df: pd.DataFrame, mode: str) -> pd.DataFrame:
    rows = []

    for _, row in parsed_df.iterrows():
        source_file = row["source_file"]
        source_path = row.get("source_path", "")
        run_type = row["run_type"]
        focal_current_product = row.get("focal_current_product", None)
        qid = int(row["question_id"])
        answer = row["answer_text"]
        seq = citation_sequence(answer)
        total_product_ref_count = len(seq)
        mention_seq = brand_mention_sequence(answer)
        total_brand_mention_count = len(mention_seq)

        unique_order = []
        for p in seq:
            if p not in unique_order:
                unique_order.append(p)

        first_unique = unique_order[0] if unique_order else None

        unique_mention_order = []
        for p in mention_seq:
            if p not in unique_mention_order:
                unique_mention_order.append(p)
        first_mentioned_unique = unique_mention_order[0] if unique_mention_order else None

        for product in PRODUCTS:
            if mode == "baseline":
                version_status = "original_2023_2024"
            else:
                version_status = "current_2026" if product == focal_current_product else "original_2023_2024"

            ref_count = seq.count(product)
            cited = ref_count > 0
            first_idx = seq.index(product) + 1 if cited else np.nan
            first_score = 1 / first_idx if cited else 0.0
            citation_share_of_voice = ref_count / total_product_ref_count if total_product_ref_count else 0.0
            citation_position_proxy = first_idx if cited else N_PRODUCTS + 1
            citation_position_score = max((N_PRODUCTS + 1 - citation_position_proxy) / N_PRODUCTS, 0)

            brand_mention_count = mention_seq.count(product)
            brand_mentioned = brand_mention_count > 0
            first_mention_idx = mention_seq.index(product) + 1 if brand_mentioned else np.nan
            first_mention_score = 1 / first_mention_idx if brand_mentioned else 0.0
            mention_share_of_voice = brand_mention_count / total_brand_mention_count if total_brand_mention_count else 0.0
            mention_position_proxy = unique_mention_order.index(product) + 1 if product in unique_mention_order else N_PRODUCTS + 1
            mention_position_score = max((N_PRODUCTS + 1 - mention_position_proxy) / N_PRODUCTS, 0)
            average_mention_position_proxy = mention_position_proxy

            recommendation_rank_proxy = unique_order.index(product) + 1 if product in unique_order else N_PRODUCTS + 1
            rank_score_proxy = max((N_PRODUCTS + 1 - recommendation_rank_proxy) / N_PRODUCTS, 0)

            cited_sents = cited_sentences_for_product(answer, product)
            mention_sents = mentioned_sentences_for_product(answer, product)
            cited_answer_text = " ".join(cited_sents)
            mentioned_answer_text = " ".join(mention_sents)
            product_text = product_texts.get((version_status, product), "")

            feature_cats = product_feature_categories(answer, product)
            feature_coverage = len(feature_cats) / len(FEATURE_PATTERNS)
            tfidf_sim = tfidf_similarity_to_product_text(product_text, cited_answer_text) if cited else 0.0
            ngram_overlap = simple_ngram_overlap(product_text, cited_answer_text, n=2) if cited else 0.0
            brand_context_text = " ".join([cited_answer_text, mentioned_answer_text]).strip()
            brand_context_sentiment_score, brand_context_positive_hits, brand_context_negative_hits = simple_brand_sentiment(brand_context_text)

            ref_component = min(ref_count / 3, 1)
            first_component = first_score

            # Product Answer Influence Score, adapted for this benchmark.
            # This is an observational proxy, not a causal or attention-based metric.
            pais = (
                0.25 * ref_component
                + 0.20 * first_component
                + 0.20 * feature_coverage
                + 0.20 * tfidf_sim
                + 0.15 * ngram_overlap
            )

            rows.append({
                "comparison": COMPARISON_NAME,
                "mode": mode,
                "source_file": source_file,
                "source_path": source_path,
                "run_type": run_type,
                "focal_current_product": focal_current_product,
                "version_status_for_product": version_status,
                "prompt_set_start": row.get("prompt_set_start", np.nan),
                "prompt_set_end": row.get("prompt_set_end", np.nan),
                "prompt_set_label": row.get("prompt_set_label", ""),
                "question_id": qid,
                "product": product,
                "cited": int(cited),
                "ref_count": ref_count,
                "first_citation_index": first_idx,
                "first_citation_score": first_score,
                "citation_position_proxy": citation_position_proxy,
                "citation_position_score": citation_position_score,
                "citation_share_of_voice": citation_share_of_voice,
                "recommendation_rank_proxy": recommendation_rank_proxy,
                "rank_score_proxy": rank_score_proxy,
                "top1_cited_flag": int(product == first_unique),
                "brand_mentioned": int(brand_mentioned),
                "brand_mention_count": brand_mention_count,
                "first_mention_index": first_mention_idx,
                "first_mention_score": first_mention_score,
                "mention_share_of_voice": mention_share_of_voice,
                "mention_position_proxy": mention_position_proxy,
                "mention_position_score": mention_position_score,
                "average_mention_position_proxy": average_mention_position_proxy,
                "top1_mentioned_flag": int(product == first_mentioned_unique),
                "feature_categories": ";".join(feature_cats),
                "feature_count": len(feature_cats),
                "feature_coverage": feature_coverage,
                "tfidf_product_answer_similarity": tfidf_sim,
                "bigram_overlap_product_answer": ngram_overlap,
                "pais_product_answer_influence_score": pais,
                "brand_context_sentiment_score": brand_context_sentiment_score,
                "brand_context_positive_hits": brand_context_positive_hits,
                "brand_context_negative_hits": brand_context_negative_hits,
                "answer_word_count": word_count(answer),
                "citation_sequence": " > ".join(seq),
                "cited_answer_sentences": cited_answer_text,
                "mentioned_answer_sentences": mentioned_answer_text,
            })

    return pd.DataFrame(rows)

baseline_qp_metrics = build_question_product_metrics(baseline_answers, mode="baseline")
current_qp_metrics = build_question_product_metrics(current_answers, mode="one_product_current")

display(baseline_qp_metrics.head(15))
display(current_qp_metrics.head(15))


,comparison,mode,source_file,source_path,run_type,focal_current_product,version_status_for_product,prompt_set_start,prompt_set_end,prompt_set_label,...,tfidf_product_answer_similarity,bigram_overlap_product_answer,pais_product_answer_influence_score,brand_context_sentiment_score,brand_context_positive_hits,brand_context_negative_hits,answer_word_count,citation_sequence,cited_answer_sentences,mentioned_answer_sentences
0,01_2023_2024_original_vs_2026_current,baseline,baseline_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\baseline\baseline_1.txt,baseline_all_original,None,original_2023_2024,1,5,1-5,...,0.000000,0.000000,0.000000,0.000000,0,0,152,Delsey > Delsey > Travelpro,,
1,01_2023_2024_original_vs_2026_current,baseline,baseline_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\baseline\baseline_1.txt,baseline_all_original,None,original_2023_2024,1,5,1-5,...,0.000000,0.000000,0.000000,0.000000,0,0,152,Delsey > Delsey > Travelpro,,
2,01_2023_2024_original_vs_2026_current,baseline,baseline_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\baseline\baseline_1.txt,baseline_all_original,None,original_2023_2024,1,5,1-5,...,0.050716,0.000000,0.160143,-0.333333,1,2,152,Delsey > Delsey > Travelpro,[Travelpro],"The Travelpro Platinum Elite Carry-On Expandable Hardside Spinner is also a strong choice because it has 45 L of volume, a 2-inch expansion option, and is sizer-bin tested for ..."
3,01_2023_2024_original_vs_2026_current,baseline,baseline_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\baseline\baseline_1.txt,baseline_all_original,None,original_2023_2024,1,5,1-5,...,0.124547,0.104651,0.502011,0.555556,7,2,152,Delsey > Delsey > Travelpro,"[Delsey]\nIt is also practical for this trip length because it has dual compartments, compression cross straps, a zippered divider, mesh pockets, and included laundry and shoe ...","For a 3-5 day trip, I would recommend the Delsey CHATELET AIR 2.0 Carry-On Plus Spinner first because the page specifically says the Carry-On Plus is ideal for 3-5 day trips, f..."
4,01_2023_2024_original_vs_2026_current,baseline,baseline_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\baseline\baseline_1.txt,baseline_all_original,None,original_2023_2024,1,5,1-5,...,0.000000,0.000000,0.000000,0.000000,0,0,152,Delsey > Delsey > Travelpro,,
5,01_2023_2024_original_vs_2026_current,baseline,baseline_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\baseline\baseline_1.txt,baseline_all_original,None,original_2023_2024,1,5,1-5,...,0.064614,0.000000,0.146256,1.000000,3,0,132,Travelpro > Travelpro > Delsey > Monos,[Monos],"Monos also looks protective, but the provided page gives fewer concrete protection details beyond an unbreakable polycarbonate shell, aerospace-grade polycarbonate, a TSA-appro..."
6,01_2023_2024_original_vs_2026_current,baseline,baseline_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\baseline\baseline_1.txt,baseline_all_original,None,original_2023_2024,1,5,1-5,...,0.000000,0.000000,0.000000,0.000000,0,0,132,Travelpro > Travelpro > Delsey > Monos,,
7,01_2023_2024_original_vs_2026_current,baseline,baseline_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\baseline\baseline_1.txt,baseline_all_original,None,original_2023_2024,1,5,1-5,...,0.098992,0.230769,0.452659,1.000000,4,0,132,Travelpro > Travelpro > Delsey > Monos,"[Travelpro]\nIt also adds aluminum corner armor for high-impact areas, a textured finish to reduce the visibility of scuffs or scratches, and a built-in TSA-approved lock. [Tra...","The Travelpro seems best if durability and protecting belongings matter most because its page describes an ultra-strong, easy-to-clean 100% polycarbo

,comparison,mode,source_file,source_path,run_type,focal_current_product,version_status_for_product,prompt_set_start,prompt_set_end,prompt_set_label,...,tfidf_product_answer_similarity,bigram_overlap_product_answer,pais_product_answer_influence_score,brand_context_sentiment_score,brand_context_positive_hits,brand_context_negative_hits,answer_word_count,citation_sequence,cited_answer_sentences,mentioned_answer_sentences
0,01_2023_2024_original_vs_2026_current,one_product_current,beis_2023vs2026_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\beis_2023vs2026_1.txt,one_product_current,BEIS,original_2023_2024,1,5,1-5,...,0.000000,0.000000,0.000000,0.000000,0,0,182,Delsey > Delsey > Travelpro > BEIS,,
1,01_2023_2024_original_vs_2026_current,one_product_current,beis_2023vs2026_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\beis_2023vs2026_1.txt,one_product_current,BEIS,current_2026,1,5,1-5,...,0.000000,0.000000,0.133333,0.000000,1,1,182,Delsey > Delsey > Travelpro > BEIS,[BEIS],"BÉIS is appealing for overpackers because it offers 49-61 L capacity and 2 inches of expandable space, but its page tells shoppers to check airline guidelines, so I would be mo..."
2,01_2023_2024_original_vs_2026_current,one_product_current,beis_2023vs2026_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\beis_2023vs2026_1.txt,one_product_current,BEIS,original_2023_2024,1,5,1-5,...,0.042411,0.026316,0.215061,-0.200000,2,3,182,Delsey > Delsey > Travelpro > BEIS,"[Travelpro] BÉIS is appealing for overpackers because it offers 49-61 L capacity and 2 inches of expandable space, but its page tells shoppers to check airline guidelines, so I...","Travelpro is a strong alternative if you want more tech and expansion because it has 45 L of volume, a 2-inch expansion option, USB A & C ports, and a sizer-bin-tested design f..."
3,01_2023_2024_original_vs_2026_current,one_product_current,beis_2023vs2026_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\beis_2023vs2026_1.txt,one_product_current,BEIS,original_2023_2024,1,5,1-5,...,0.105548,0.130435,0.481026,0.333333,4,2,182,Delsey > Delsey > Travelpro > BEIS,"[Delsey] It also has 44 L of volume, dual compartments with compression cross straps, a zippered divider, mesh pockets, and included laundry and shoe bags, so it gives enough s...","For a 3-5 day trip, I would recommend the Delsey CHATELET AIR 2.0 Carry-On Plus Spinner as the most directly matched option because its page specifically says the Carry-On Plus..."
4,01_2023_2024_original_vs_2026_current,one_product_current,beis_2023vs2026_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\beis_2023vs2026_1.txt,one_product_current,BEIS,original_2023_2024,1,5,1-5,...,0.000000,0.000000,0.000000,0.000000,0,0,182,Delsey > Delsey > Travelpro > BEIS,,
5,01_2023_2024_original_vs_2026_current,one_product_current,beis_2023vs2026_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\beis_2023vs2026_1.txt,one_product_current,BEIS,original_2023_2024,1,5,1-5,...,0.064614,0.000000,0.136256,0.500000,3,1,183,Travelpro > Travelpro > Delsey > Delsey > Monos,[Monos],"Monos is also durable on paper because it lists an unbreakable polycarbonate shell, an aerospace-grade polycarbonate shell, a TSA-approved lock, reverse coil YKK zippers, and a..."
6,01_2023_2024_original_vs_2026_current,one_product_current,beis_2023vs2026_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\beis_2023vs2026_1.txt,one_product_current,BEIS,current

## Match current runs to baseline question IDs

This is important for multi-prompt analysis.

If current files include Q1–5, Q6–10, and Q16–20, then the baseline comparison should use only those same question IDs.  
If Q11–15 GEO files are accidentally in the current folder, they are ignored in Comparison 01.


In [8]:
# =============================
# Matched current-vs-baseline question-product deltas
# =============================

METRIC_COLS = [
    "cited",
    "ref_count",
    "first_citation_score",
    "recommendation_rank_proxy",
    "rank_score_proxy",
    "top1_cited_flag",
    "feature_count",
    "feature_coverage",
    "tfidf_product_answer_similarity",
    "bigram_overlap_product_answer",
    "pais_product_answer_influence_score",
    "brand_mentioned",
    "brand_mention_count",
    "mention_share_of_voice",
    "citation_share_of_voice",
    "first_mention_score",
    "mention_position_proxy",
    "mention_position_score",
    "average_mention_position_proxy",
    "citation_position_proxy",
    "citation_position_score",
    "brand_context_sentiment_score",
    "brand_context_positive_hits",
    "brand_context_negative_hits",
]

current_focal_rows = current_qp_metrics[
    current_qp_metrics["product"] == current_qp_metrics["focal_current_product"]
].copy()

baseline_lookup_cols = [
    "product",
    "question_id",
    "source_file",
    "prompt_set_label",
] + METRIC_COLS

baseline_lookup = baseline_qp_metrics[baseline_lookup_cols].copy()

# If multiple baseline files contain the same product/question_id, average them.
# This supports repeated baseline runs while keeping the comparison stable.
agg_dict = {col: "mean" for col in METRIC_COLS}
agg_dict.update({
    "source_file": lambda x: ";".join(sorted(set(map(str, x)))),
    "prompt_set_label": lambda x: ";".join(sorted(set(map(str, x)))),
})
baseline_lookup = (
    baseline_lookup
    .groupby(["product", "question_id"], as_index=False)
    .agg(agg_dict)
)

matched = current_focal_rows.merge(
    baseline_lookup,
    on=["product", "question_id"],
    how="left",
    suffixes=("_current", "_baseline"),
)

matched["baseline_match_found"] = matched["source_file_baseline"].notna()

missing_baseline_matches = matched[~matched["baseline_match_found"]].copy()
current_question_ids = set(map(int, current_focal_rows["question_id"].unique()))
baseline_question_ids = set(map(int, baseline_lookup["question_id"].unique()))
matched_question_ids = set(map(int, matched[matched["baseline_match_found"]]["question_id"].unique()))
missing_baseline_question_ids = sorted(current_question_ids - baseline_question_ids)
missing_current_question_ids_vs_expected = sorted(set(EXPECTED_QUESTION_IDS) - current_question_ids)
missing_baseline_question_ids_vs_expected = sorted(set(EXPECTED_QUESTION_IDS) - baseline_question_ids)

matching_coverage_summary = pd.DataFrame([{
    "n_current_focal_rows": len(current_focal_rows),
    "n_matched_rows": int(matched["baseline_match_found"].sum()),
    "n_unmatched_current_rows": int((~matched["baseline_match_found"]).sum()),
    "current_question_ids": ",".join(map(str, sorted(current_question_ids))),
    "baseline_question_ids": ",".join(map(str, sorted(baseline_question_ids))),
    "matched_question_ids": ",".join(map(str, sorted(matched_question_ids))),
    "missing_baseline_question_ids_for_current": ",".join(map(str, missing_baseline_question_ids)),
    "missing_current_question_ids_vs_expected": ",".join(map(str, missing_current_question_ids_vs_expected)),
    "missing_baseline_question_ids_vs_expected": ",".join(map(str, missing_baseline_question_ids_vs_expected)),
}])

print("Matching coverage summary:")
display(matching_coverage_summary)

if not missing_baseline_matches.empty:
    print("WARNING: Some current focal rows do not have matching baseline question IDs.")
    display(
        missing_baseline_matches[
            ["source_file_current", "focal_current_product", "product", "question_id", "prompt_set_label_current"]
        ].sort_values(["product", "question_id", "source_file_current"])
    )

if missing_current_question_ids_vs_expected:
    print("NOTE: These expected question IDs are not present in included current-vs-2026 files:", missing_current_question_ids_vs_expected)
    print("If Q11-15 files are currently named with 'vsGEO', they are intentionally ignored in Comparison 01. Rename them to '*2023vs2026_11.txt' only if they are truly current-vs-2026 runs.")

if (not ALLOW_PARTIAL_MATCH) and missing_baseline_question_ids:
    raise ValueError(
        "Cannot compute a full multi-prompt current-vs-baseline delta because some current question IDs do not have matching all-original baseline answers. "
        f"Missing baseline question IDs for current files: {missing_baseline_question_ids}. "
        "Add all-original baseline answer files for those questions to the baseline folder, or set ALLOW_PARTIAL_MATCH = True if you intentionally want a partial matched analysis."
    )

matched_complete = matched[matched["baseline_match_found"]].copy()

for col in METRIC_COLS:
    matched_complete[f"delta_{col}"] = matched_complete[f"{col}_current"] - matched_complete[f"{col}_baseline"]

# More readable aliases for downstream tables.
matched_complete["delta_citation_rate"] = matched_complete["delta_cited"]
matched_complete["delta_mean_ref_count"] = matched_complete["delta_ref_count"]
matched_complete["rank_improvement_current_minus_baseline"] = (
    matched_complete["recommendation_rank_proxy_baseline"] - matched_complete["recommendation_rank_proxy_current"]
)
matched_complete["delta_top1_cited_share"] = matched_complete["delta_top1_cited_flag"]
matched_complete["delta_pais"] = matched_complete["delta_pais_product_answer_influence_score"]

print("Current focal rows:", len(current_focal_rows))
print("Matched rows used for delta:", len(matched_complete))
display(matched_complete.head(30))


Matching coverage summary:


,n_current_focal_rows,n_matched_rows,n_unmatched_current_rows,current_question_ids,baseline_question_ids,matched_question_ids,missing_baseline_question_ids_for_current,missing_current_question_ids_vs_expected,missing_baseline_question_ids_vs_expected
0,100,100,0,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20","1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20","1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",,,


Current focal rows: 100
Matched rows used for delta: 100


,comparison,mode,source_file_current,source_path,run_type,focal_current_product,version_status_for_product,prompt_set_start,prompt_set_end,prompt_set_label_current,...,delta_citation_position_proxy,delta_citation_position_score,delta_brand_context_sentiment_score,delta_brand_context_positive_hits,delta_brand_context_negative_hits,delta_citation_rate,delta_mean_ref_count,rank_improvement_current_minus_baseline,delta_top1_cited_share,delta_pais
0,01_2023_2024_original_vs_2026_current,one_product_current,beis_2023vs2026_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\beis_2023vs2026_1.txt,one_product_current,BEIS,current_2026,1,5,1-5,...,-2.0,0.4,0.000000,1.0,1.0,1.0,1.0,3.0,0.0,0.133333
1,01_2023_2024_original_vs_2026_current,one_product_current,beis_2023vs2026_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\beis_2023vs2026_1.txt,one_product_current,BEIS,current_2026,1,5,1-5,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
2,01_2023_2024_original_vs_2026_current,one_product_current,beis_2023vs2026_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\beis_2023vs2026_1.txt,one_product_current,BEIS,current_2026,1,5,1-5,...,1.0,-0.2,0.000000,-3.0,0.0,0.0,0.0,-2.0,0.0,-0.039413
3,01_2023_2024_original_vs_2026_current,one_product_current,beis_2023vs2026_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\beis_2023vs2026_1.txt,one_product_current,BEIS,current_2026,1,5,1-5,...,0.0,0.0,0.000000,1.0,0.0,0.0,0.0,0.0,0.0,0.038268
4,01_2023_2024_original_vs_2026_current,one_product_current,beis_2023vs2026_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\beis_2023vs2026_1.txt,one_product_current,BEIS,current_2026,1,5,1-5,...,0.0,0.0,0.533333,5.0,2.0,1.0,1.0,2.0,0.0,0.166057
5,01_2023_2024_original_vs_2026_current,one_product_current,beis_2023vs2026_11.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\beis_2023vs2026_11.txt,one_product_current,BEIS,current_2026,11,15,11-15,...,0.0,0.0,1.000000,0.0,-2.0,0.0,0.0,0.0,0.0,-0.040942
6,01_2023_2024_original_vs_2026_current,one_product_current,beis_2023vs2026_11.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\beis_2023vs2026_11.txt,one_product_current,BEIS,current_2026,11,15,11-15,...,-1.0,0.2,0.333333,-3.0,-1.0,0.0,-1.0,1.0,0.0,-0.086776
7,01_2023_2024_original_vs_2026_current,one_product_current,beis_2023vs2026_11.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\beis_2023vs2026_11.txt,one_product_current,BEIS,current_2026,11,15,11-15,...,1.0,-0.2,-0.380952,-1.0,2.0,0.0,0.0,-1.0,0.0,-0.039160
8,01_2023_2024_original_vs_2026_current,one_product_current,beis_2023vs2026_11.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\beis_2023vs2026_11.txt,one_product_current,BEIS,current_2026,11,15,11-15,...,-1.0,0.2,0.000000,0.0,0.0,0.0,-1.0,1.0,1.0,-0.017498
9,01_2023_2024_original_vs_2026_current,one_product_current,beis_2023vs2026_11.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\beis_2023vs2026_11.txt,one_product_current,BEIS,current_2026,11,15,11-15,...,1.0,-0.2,-0.166667,-7.0,-1.0,0.0,0.0,-1.0,0.0,-0.061212


## Product summaries

The main product-level summary is now based on **matched current-vs-baseline rows**.

This prevents unfair comparison if some prompt sets exist for current files but not in the baseline files, or vice versa.


In [9]:
# =============================
# Product-level summaries from matched rows
# =============================

def summarize_side_from_matched(matched_df: pd.DataFrame, side: str, label: str) -> pd.DataFrame:
    suffix = "_current" if side == "current" else "_baseline"

    temp = pd.DataFrame({
        "product": matched_df["product"],
        "question_id": matched_df["question_id"],
        "source_file_current": matched_df["source_file_current"],
        "source_file_baseline": matched_df["source_file_baseline"],
        "prompt_set_label_current": matched_df["prompt_set_label_current"],
        "cited": matched_df[f"cited{suffix}"],
        "ref_count": matched_df[f"ref_count{suffix}"],
        "first_citation_score": matched_df[f"first_citation_score{suffix}"],
        "recommendation_rank_proxy": matched_df[f"recommendation_rank_proxy{suffix}"],
        "rank_score_proxy": matched_df[f"rank_score_proxy{suffix}"],
        "top1_cited_flag": matched_df[f"top1_cited_flag{suffix}"],
        "feature_count": matched_df[f"feature_count{suffix}"],
        "feature_coverage": matched_df[f"feature_coverage{suffix}"],
        "tfidf_product_answer_similarity": matched_df[f"tfidf_product_answer_similarity{suffix}"],
        "bigram_overlap_product_answer": matched_df[f"bigram_overlap_product_answer{suffix}"],
        "pais_product_answer_influence_score": matched_df[f"pais_product_answer_influence_score{suffix}"],
    })

    summary = (
        temp.groupby("product")
        .agg(
            n_question_contexts=("question_id", "count"),
            question_ids=("question_id", lambda x: ",".join(map(str, sorted(set(x))))),
            n_current_source_files=("source_file_current", "nunique"),
            n_baseline_source_files=("source_file_baseline", "nunique"),
            prompt_sets=("prompt_set_label_current", lambda x: join_prompt_set_labels(x)),
            citation_rate=("cited", "mean"),
            mean_ref_count=("ref_count", "mean"),
            total_ref_count=("ref_count", "sum"),
            mean_first_citation_score=("first_citation_score", "mean"),
            mean_recommendation_rank_proxy=("recommendation_rank_proxy", "mean"),
            mean_rank_score_proxy=("rank_score_proxy", "mean"),
            top1_cited_share=("top1_cited_flag", "mean"),
            mean_feature_count=("feature_count", "mean"),
            mean_feature_coverage=("feature_coverage", "mean"),
            mean_tfidf_product_answer_similarity=("tfidf_product_answer_similarity", "mean"),
            mean_bigram_overlap_product_answer=("bigram_overlap_product_answer", "mean"),
            mean_pais_product_answer_influence_score=("pais_product_answer_influence_score", "mean"),
        )
        .reset_index()
    )
    summary.insert(0, "metric_source", label)
    return summary

baseline_product_summary_matched = summarize_side_from_matched(
    matched_complete,
    side="baseline",
    label="baseline_all_original_matched_questions",
)

current_product_summary_matched = summarize_side_from_matched(
    matched_complete,
    side="current",
    label="one_product_current_focal_matched_questions",
)

display(baseline_product_summary_matched)
display(current_product_summary_matched)


,metric_source,product,n_question_contexts,question_ids,n_current_source_files,n_baseline_source_files,prompt_sets,citation_rate,mean_ref_count,total_ref_count,mean_first_citation_score,mean_recommendation_rank_proxy,mean_rank_score_proxy,top1_cited_share,mean_feature_count,mean_feature_coverage,mean_tfidf_product_answer_similarity,mean_bigram_overlap_product_answer,mean_pais_product_answer_influence_score
0,baseline_all_original_matched_questions,BEIS,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.70,1.15,23.0,0.362500,3.30,0.54,0.20,3.60,0.189474,0.027449,0.024691,0.215421
1,baseline_all_original_matched_questions,Delsey,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.95,1.50,30.0,0.383889,2.70,0.66,0.15,4.40,0.231579,0.058741,0.058481,0.268614
2,baseline_all_original_matched_questions,INUSA,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.65,1.05,21.0,0.163095,4.70,0.26,0.05,2.65,0.139474,0.026307,0.016567,0.155760
3,baseline_all_original_matched_questions,Monos,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.75,1.05,21.0,0.197560,4.20,0.36,0.05,3.35,0.176316,0.034622,0.016093,0.171613
4,baseline_all_original_matched_questions,Travelpro,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.90,1.90,38.0,0.683333,1.95,0.81,0.55,4.20,0.221053,0.069550,0.117828,0.358295


,metric_source,product,n_question_contexts,question_ids,n_current_source_files,n_baseline_source_files,prompt_sets,citation_rate,mean_ref_count,total_ref_count,mean_first_citation_score,mean_recommendation_rank_proxy,mean_rank_score_proxy,top1_cited_share,mean_feature_count,mean_feature_coverage,mean_tfidf_product_answer_similarity,mean_bigram_overlap_product_answer,mean_pais_product_answer_influence_score
0,one_product_current_focal_matched_questions,BEIS,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.85,1.25,25,0.443333,3.00,0.60,0.30,3.35,0.176316,0.035131,0.049572,0.242559
1,one_product_current_focal_matched_questions,Delsey,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.85,1.25,25,0.297143,3.20,0.56,0.10,3.75,0.197368,0.040534,0.027863,0.211189
2,one_product_current_focal_matched_questions,INUSA,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.65,0.80,16,0.195893,4.40,0.32,0.05,2.50,0.131579,0.024324,0.018646,0.139823
3,one_product_current_focal_matched_questions,Monos,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.85,1.25,25,0.504643,2.70,0.66,0.35,3.80,0.200000,0.046023,0.057225,0.262884
4,one_product_current_focal_matched_questions,Travelpro,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.90,2.00,40,0.783333,1.75,0.85,0.70,5.55,0.292105,0.094683,0.140161,0.405049


## Current-vs-baseline delta by product

This is the main output.

Positive delta means the product became more visible or more prominent when its own page was replaced by the 2026 current version, compared with the matched all-original baseline.


In [10]:
# =============================
# Current - baseline product-level delta
# =============================

def make_current_vs_baseline_delta_from_matched(matched_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for product, g in matched_df.groupby("product"):
        rows.append({
            "comparison": COMPARISON_NAME,
            "product": product,
            "matched_n_question_contexts": int(len(g)),
            "question_ids": ",".join(map(str, sorted(g["question_id"].unique()))),
            "current_source_files": ";".join(sorted(g["source_file_current"].unique())),
            "baseline_source_files": ";".join(sorted(g["source_file_baseline"].unique())),
            "prompt_sets": join_prompt_set_labels(g["prompt_set_label_current"]),

            "baseline_citation_rate": g["cited_baseline"].mean(),
            "current_citation_rate": g["cited_current"].mean(),
            "delta_citation_rate": g["delta_cited"].mean(),

            "baseline_mean_ref_count": g["ref_count_baseline"].mean(),
            "current_mean_ref_count": g["ref_count_current"].mean(),
            "delta_mean_ref_count": g["delta_ref_count"].mean(),

            "baseline_first_citation_score": g["first_citation_score_baseline"].mean(),
            "current_first_citation_score": g["first_citation_score_current"].mean(),
            "delta_first_citation_score": g["delta_first_citation_score"].mean(),

            "baseline_mean_recommendation_rank_proxy": g["recommendation_rank_proxy_baseline"].mean(),
            "current_mean_recommendation_rank_proxy": g["recommendation_rank_proxy_current"].mean(),
            "rank_improvement_current_minus_baseline": (
                g["recommendation_rank_proxy_baseline"].mean() - g["recommendation_rank_proxy_current"].mean()
            ),

            "baseline_rank_score_proxy": g["rank_score_proxy_baseline"].mean(),
            "current_rank_score_proxy": g["rank_score_proxy_current"].mean(),
            "delta_rank_score_proxy": g["delta_rank_score_proxy"].mean(),

            "baseline_top1_cited_share": g["top1_cited_flag_baseline"].mean(),
            "current_top1_cited_share": g["top1_cited_flag_current"].mean(),
            "delta_top1_cited_share": g["delta_top1_cited_flag"].mean(),

            "baseline_feature_coverage": g["feature_coverage_baseline"].mean(),
            "current_feature_coverage": g["feature_coverage_current"].mean(),
            "delta_feature_coverage": g["delta_feature_coverage"].mean(),

            "baseline_pais": g["pais_product_answer_influence_score_baseline"].mean(),
            "current_pais": g["pais_product_answer_influence_score_current"].mean(),
            "delta_pais": g["delta_pais_product_answer_influence_score"].mean(),
        })

    delta = pd.DataFrame(rows)

    # Composite descriptive score.
    # This is not a causal estimate; it summarizes observed answer-behavior movement.
    delta["current_page_advantage_score"] = (
        0.25 * delta["delta_rank_score_proxy"]
        + 0.20 * delta["delta_citation_rate"]
        + 0.20 * delta["delta_first_citation_score"]
        + 0.15 * delta["delta_top1_cited_share"]
        + 0.10 * delta["delta_feature_coverage"]
        + 0.10 * delta["delta_pais"]
    )

    delta["direction_label"] = np.select(
        [
            delta["current_page_advantage_score"] > 0.05,
            delta["current_page_advantage_score"] < -0.05,
        ],
        [
            "current_2026_stronger",
            "baseline_2023_2024_stronger",
        ],
        default="similar_or_small_change",
    )

    return delta.sort_values("current_page_advantage_score", ascending=False)

current_vs_baseline_delta_by_product = make_current_vs_baseline_delta_from_matched(matched_complete)

display(current_vs_baseline_delta_by_product)


,comparison,product,matched_n_question_contexts,question_ids,current_source_files,baseline_source_files,prompt_sets,baseline_citation_rate,current_citation_rate,delta_citation_rate,...,current_top1_cited_share,delta_top1_cited_share,baseline_feature_coverage,current_feature_coverage,delta_feature_coverage,baseline_pais,current_pais,delta_pais,current_page_advantage_score,direction_label
3,01_2023_2024_original_vs_2026_current,Monos,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",monos_2023vs2026_1.txt;monos_2023vs2026_11.txt;monos_2023vs2026_16.txt;monos_2023vs2026_6.txt,baseline_1.txt;baseline_11.txt;baseline_16.txt;baseline_6.txt,1-5;6-10;11-15;16-20,0.75,0.85,0.10,...,0.35,0.30,0.176316,0.200000,0.023684,0.171613,0.262884,0.091270,0.212912,current_2026_stronger
0,01_2023_2024_original_vs_2026_current,BEIS,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",beis_2023vs2026_1.txt;beis_2023vs2026_11.txt;beis_2023vs2026_16.txt;beis_2023vs2026_6.txt,baseline_1.txt;baseline_11.txt;baseline_16.txt;baseline_6.txt,1-5;6-10;11-15;16-20,0.70,0.85,0.15,...,0.30,0.10,0.189474,0.176316,-0.013158,0.215421,0.242559,0.027137,0.077565,current_2026_stronger
4,01_2023_2024_original_vs_2026_current,Travelpro,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",travel_2023vs2026_1.txt;travel_2023vs2026_11.txt.txt;travel_2023vs2026_16.txt;travel_2023vs2026_6.txt,baseline_1.txt;baseline_11.txt;baseline_16.txt;baseline_6.txt,1-5;6-10;11-15;16-20,0.90,0.90,0.00,...,0.70,0.15,0.221053,0.292105,0.071053,0.358295,0.405049,0.046754,0.064281,current_2026_stronger
2,01_2023_2024_original_vs_2026_current,INUSA,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",inusa_2023vs2026_1.txt;inusa_2023vs2026_11.txt;inusa_2023vs2026_16.txt;inusea_2023vs2026_6.txt,baseline_1.txt;baseline_11.txt;baseline_16.txt;baseline_6.txt,1-5;6-10;11-15;16-20,0.65,0.65,0.00,...,0.05,0.00,0.139474,0.131579,-0.007895,0.155760,0.139823,-0.015938,0.019176,similar_or_small_change
1,01_2023_2024_original_vs_2026_current,Delsey,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",delsey_2023vs2026_1.txt;delsey_2023vs2026_11.txt;delsey_2023vs2026_6.txt;desley_2023vs2026_16.txt,baseline_1.txt;baseline_11.txt;baseline_16.txt;baseline_6.txt,1-5;6-10;11-15;16-20,0.95,0.85,-0.10,...,0.10,-0.05,0.231579,0.197368,-0.034211,0.268614,0.211189,-0.057425,-0.079013,baseline_2023_2024_stronger


## Overall summary


In [11]:
overall_current_vs_baseline_summary = pd.DataFrame([{
    "comparison": COMPARISON_NAME,
    "purpose": "Check how much the actual product page changed over time",
    "interpretation": "General website/text change",
    "baseline_dir": str(BASELINE_DIR),
    "comparison_dir": str(COMPARISON_DIR),
    "n_baseline_files": len(BASELINE_FILES),
    "n_included_current_files": len(TXT_FILES),
    "n_ignored_txt_files": len(IGNORED_TXT_FILES),
    "included_question_ids": ",".join(map(str, sorted(matched_complete["question_id"].unique()))),
    "baseline_question_ids": ",".join(map(str, sorted(baseline_question_ids))),
    "current_question_ids": ",".join(map(str, sorted(current_question_ids))),
    "missing_baseline_question_ids_for_current": ",".join(map(str, missing_baseline_question_ids)),
    "missing_current_question_ids_vs_expected": ",".join(map(str, missing_current_question_ids_vs_expected)),
    "n_products": len(current_vs_baseline_delta_by_product),
    "mean_delta_citation_rate": current_vs_baseline_delta_by_product["delta_citation_rate"].mean(),
    "mean_delta_rank_score_proxy": current_vs_baseline_delta_by_product["delta_rank_score_proxy"].mean(),
    "mean_rank_improvement_current_minus_baseline": current_vs_baseline_delta_by_product["rank_improvement_current_minus_baseline"].mean(),
    "mean_delta_first_citation_score": current_vs_baseline_delta_by_product["delta_first_citation_score"].mean(),
    "mean_delta_top1_cited_share": current_vs_baseline_delta_by_product["delta_top1_cited_share"].mean(),
    "mean_delta_feature_coverage": current_vs_baseline_delta_by_product["delta_feature_coverage"].mean(),
    "mean_delta_pais": current_vs_baseline_delta_by_product["delta_pais"].mean(),
    "mean_current_page_advantage_score": current_vs_baseline_delta_by_product["current_page_advantage_score"].mean(),
    "n_current_2026_stronger": int((current_vs_baseline_delta_by_product["direction_label"] == "current_2026_stronger").sum()),
    "n_baseline_2023_2024_stronger": int((current_vs_baseline_delta_by_product["direction_label"] == "baseline_2023_2024_stronger").sum()),
    "n_similar_or_small_change": int((current_vs_baseline_delta_by_product["direction_label"] == "similar_or_small_change").sum()),
}])

display(overall_current_vs_baseline_summary)


,comparison,purpose,interpretation,baseline_dir,comparison_dir,n_baseline_files,n_included_current_files,n_ignored_txt_files,included_question_ids,baseline_question_ids,...,mean_delta_rank_score_proxy,mean_rank_improvement_current_minus_baseline,mean_delta_first_citation_score,mean_delta_top1_cited_share,mean_delta_feature_coverage,mean_delta_pais,mean_current_page_advantage_score,n_current_2026_stronger,n_baseline_2023_2024_stronger,n_similar_or_small_change
0,01_2023_2024_original_vs_2026_current,Check how much the actual product page changed over time,General website/text change,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\baseline,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current,4,20,0,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20","1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",...,0.072,0.36,0.086794,0.1,0.007895,0.01836,0.058984,3,1,1


## Provider-style answer visibility metrics

In [12]:

# =============================
# Provider-style answer visibility metrics
# =============================
# These metrics translate the product-label answer files into commercial AI-visibility language.
# source/domain share is intentionally not computed here because the prompt uses controlled
# product-label citations such as [Monos], not external URL/domain citations.

PROVIDER_STYLE_METRIC_NOTE = (
    "Provider-style answer metrics added: brand_mention_rate, mention_share_of_voice, "
    "citation_share_of_voice, average_mention_position, average_citation_position, "
    "first_citation_score, sentiment context proxy, and comparison-level visibility delta. "
    "source/domain share is not available in this controlled product-label citation setting."
)

def _safe_mean(series):
    return pd.to_numeric(series, errors="coerce").mean()

def _provider_delta_rows(matched_df, left_suffix, right_suffix, left_name, right_name, improvement_label):
    rows = []
    for product, g in matched_df.groupby("product"):
        left_mention_pos = _safe_mean(g[f"average_mention_position_proxy{left_suffix}"])
        right_mention_pos = _safe_mean(g[f"average_mention_position_proxy{right_suffix}"])
        left_citation_pos = _safe_mean(g[f"citation_position_proxy{left_suffix}"])
        right_citation_pos = _safe_mean(g[f"citation_position_proxy{right_suffix}"])

        rows.append({
            "product": product,
            f"{left_name}_brand_mention_rate": _safe_mean(g[f"brand_mentioned{left_suffix}"]),
            f"{right_name}_brand_mention_rate": _safe_mean(g[f"brand_mentioned{right_suffix}"]),
            "delta_brand_mention_rate": _safe_mean(g[f"brand_mentioned{right_suffix}"] - g[f"brand_mentioned{left_suffix}"]),

            f"{left_name}_mention_share_of_voice": _safe_mean(g[f"mention_share_of_voice{left_suffix}"]),
            f"{right_name}_mention_share_of_voice": _safe_mean(g[f"mention_share_of_voice{right_suffix}"]),
            "delta_mention_share_of_voice": _safe_mean(g[f"mention_share_of_voice{right_suffix}"] - g[f"mention_share_of_voice{left_suffix}"]),

            f"{left_name}_citation_share_of_voice": _safe_mean(g[f"citation_share_of_voice{left_suffix}"]),
            f"{right_name}_citation_share_of_voice": _safe_mean(g[f"citation_share_of_voice{right_suffix}"]),
            "delta_citation_share_of_voice": _safe_mean(g[f"citation_share_of_voice{right_suffix}"] - g[f"citation_share_of_voice{left_suffix}"]),

            f"{left_name}_average_mention_position": left_mention_pos,
            f"{right_name}_average_mention_position": right_mention_pos,
            f"mention_position_improvement_{improvement_label}": left_mention_pos - right_mention_pos,

            f"{left_name}_average_citation_position": left_citation_pos,
            f"{right_name}_average_citation_position": right_citation_pos,
            f"citation_position_improvement_{improvement_label}": left_citation_pos - right_citation_pos,

            f"{left_name}_first_mention_score": _safe_mean(g[f"first_mention_score{left_suffix}"]),
            f"{right_name}_first_mention_score": _safe_mean(g[f"first_mention_score{right_suffix}"]),
            "delta_first_mention_score": _safe_mean(g[f"first_mention_score{right_suffix}"] - g[f"first_mention_score{left_suffix}"]),

            f"{left_name}_brand_context_sentiment_score": _safe_mean(g[f"brand_context_sentiment_score{left_suffix}"]),
            f"{right_name}_brand_context_sentiment_score": _safe_mean(g[f"brand_context_sentiment_score{right_suffix}"]),
            "delta_brand_context_sentiment_score": _safe_mean(g[f"brand_context_sentiment_score{right_suffix}"] - g[f"brand_context_sentiment_score{left_suffix}"]),

            "source_domain_share_status": "not_available_controlled_product_label_citations_only",
        })
    out = pd.DataFrame(rows)
    out["provider_style_visibility_delta"] = (
        0.20 * out["delta_brand_mention_rate"]
        + 0.25 * out["delta_mention_share_of_voice"]
        + 0.25 * out["delta_citation_share_of_voice"]
        + 0.15 * (out[f"mention_position_improvement_{improvement_label}"] / N_PRODUCTS)
        + 0.15 * (out[f"citation_position_improvement_{improvement_label}"] / N_PRODUCTS)
    )
    return out

provider_answer_metric_delta_by_product = _provider_delta_rows(
    matched_complete,
    left_suffix="_baseline",
    right_suffix="_current",
    left_name="baseline",
    right_name="current",
    improvement_label="current_minus_baseline",
)
current_vs_baseline_delta_by_product = current_vs_baseline_delta_by_product.merge(
    provider_answer_metric_delta_by_product,
    on="product",
    how="left",
)
current_vs_baseline_delta_by_product["historical_visibility_delta"] = current_vs_baseline_delta_by_product["current_page_advantage_score"]

# Update overall summary so the saved overall CSV includes the provider-style layer.
overall_current_vs_baseline_summary["provider_style_metric_note"] = PROVIDER_STYLE_METRIC_NOTE
for _col in [
    "delta_brand_mention_rate",
    "delta_mention_share_of_voice",
    "delta_citation_share_of_voice",
    "provider_style_visibility_delta",
    "historical_visibility_delta",
    "delta_brand_context_sentiment_score",
]:
    overall_current_vs_baseline_summary[f"mean_{_col}"] = current_vs_baseline_delta_by_product[_col].mean()

display(current_vs_baseline_delta_by_product[[
    "product",
    "delta_brand_mention_rate",
    "delta_mention_share_of_voice",
    "delta_citation_share_of_voice",
    "mention_position_improvement_current_minus_baseline",
    "citation_position_improvement_current_minus_baseline",
    "delta_brand_context_sentiment_score",
    "provider_style_visibility_delta",
    "historical_visibility_delta",
]])


,product,delta_brand_mention_rate,delta_mention_share_of_voice,delta_citation_share_of_voice,mention_position_improvement_current_minus_baseline,citation_position_improvement_current_minus_baseline,delta_brand_context_sentiment_score,provider_style_visibility_delta,historical_visibility_delta
0,Monos,0.05,0.031133,0.018648,1.40,2.00,0.156484,0.124445,0.212912
1,BEIS,0.10,-0.014153,0.018066,0.20,0.25,0.219203,0.034478,0.077565
2,Travelpro,0.00,0.001687,0.021097,0.20,0.40,-0.097425,0.023696,0.064281
3,INUSA,0.00,-0.015261,-0.013849,0.40,0.50,0.107103,0.019722,0.019176
4,Delsey,-0.15,-0.046520,-0.048175,-0.65,-0.60,-0.178175,-0.091174,-0.079013


## Save CSV outputs


In [13]:
file_manifest_comparison01.to_csv(OUTPUT_DIR / "file_manifest_comparison01.csv", index=False, encoding="utf-8-sig")
baseline_answers.to_csv(OUTPUT_DIR / "baseline_parsed_answers.csv", index=False, encoding="utf-8-sig")
current_answers.to_csv(OUTPUT_DIR / "current_parsed_answers_by_file.csv", index=False, encoding="utf-8-sig")
coverage_by_file.to_csv(OUTPUT_DIR / "question_coverage_by_file.csv", index=False, encoding="utf-8-sig")
question_coverage_summary.to_csv(OUTPUT_DIR / "question_coverage_summary.csv", index=False, encoding="utf-8-sig")
matching_coverage_summary.to_csv(OUTPUT_DIR / "matching_coverage_summary.csv", index=False, encoding="utf-8-sig")
missing_baseline_matches.to_csv(OUTPUT_DIR / "missing_baseline_matches.csv", index=False, encoding="utf-8-sig")
baseline_qp_metrics.to_csv(OUTPUT_DIR / "baseline_question_product_metrics.csv", index=False, encoding="utf-8-sig")
current_qp_metrics.to_csv(OUTPUT_DIR / "current_question_product_metrics.csv", index=False, encoding="utf-8-sig")
matched_complete.to_csv(OUTPUT_DIR / "matched_current_baseline_question_product_delta.csv", index=False, encoding="utf-8-sig")
baseline_product_summary_matched.to_csv(OUTPUT_DIR / "baseline_product_summary_matched.csv", index=False, encoding="utf-8-sig")
current_product_summary_matched.to_csv(OUTPUT_DIR / "current_product_summary_matched.csv", index=False, encoding="utf-8-sig")
current_vs_baseline_delta_by_product.to_csv(OUTPUT_DIR / "current_vs_baseline_delta_by_product.csv", index=False, encoding="utf-8-sig")
provider_answer_metric_delta_by_product.to_csv(OUTPUT_DIR / "provider_answer_metric_delta_by_product.csv", index=False, encoding="utf-8-sig")
overall_current_vs_baseline_summary.to_csv(OUTPUT_DIR / "overall_current_vs_baseline_summary.csv", index=False, encoding="utf-8-sig")
product_text_load_status.to_csv(OUTPUT_DIR / "product_text_load_status.csv", index=False, encoding="utf-8-sig")

print("Saved CSV files to:")
print(OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", p.name)


Saved CSV files to:
C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\tables\01_2023_2024_original_vs_2026_current_baseline_based
- baseline_parsed_answers.csv
- baseline_product_summary.csv
- baseline_product_summary_matched.csv
- baseline_question_product_metrics.csv
- current_parsed_answers_by_file.csv
- current_product_summary.csv
- current_product_summary_matched.csv
- current_question_product_metrics.csv
- current_vs_baseline_delta_by_product.csv
- file_manifest_comparison01.csv
- matched_current_baseline_question_product_delta.csv
- matching_coverage_summary.csv
- missing_baseline_matches.csv
- overall_current_vs_baseline_summary.csv
- product_text_load_status.csv
- provider_answer_metric_delta_by_product.csv
- question_coverage_by_file.csv
- question_coverage_summary.csv


## Quick interpretation table

Use this table first when writing the result.


In [14]:

cols = [
    "product",
    "direction_label",
    "current_page_advantage_score",
    "provider_style_visibility_delta",
    "historical_visibility_delta",
    "delta_rank_score_proxy",
    "delta_citation_rate",
    "delta_brand_mention_rate",
    "delta_citation_share_of_voice",
    "delta_mention_share_of_voice",
    "mention_position_improvement_current_minus_baseline",
    "citation_position_improvement_current_minus_baseline",
    "delta_first_citation_score",
    "delta_top1_cited_share",
    "delta_feature_coverage",
    "delta_pais",
    "matched_n_question_contexts",
    "question_ids",
    "prompt_sets",
]
cols = [c for c in cols if c in current_vs_baseline_delta_by_product.columns]
print("Quick check: provider-style answer metrics are now included alongside the original advantage metrics.")
print("source/domain share is not computed because these runs use controlled product-label citations, not external URLs/domains.")
display(current_vs_baseline_delta_by_product[cols])


Quick check: provider-style answer metrics are now included alongside the original advantage metrics.
source/domain share is not computed because these runs use controlled product-label citations, not external URLs/domains.


,product,direction_label,current_page_advantage_score,provider_style_visibility_delta,historical_visibility_delta,delta_rank_score_proxy,delta_citation_rate,delta_brand_mention_rate,delta_citation_share_of_voice,delta_mention_share_of_voice,mention_position_improvement_current_minus_baseline,citation_position_improvement_current_minus_baseline,delta_first_citation_score,delta_top1_cited_share,delta_feature_coverage,delta_pais,matched_n_question_contexts,question_ids,prompt_sets
0,Monos,current_2026_stronger,0.212912,0.124445,0.212912,0.30,0.10,0.05,0.018648,0.031133,1.40,2.00,0.307083,0.30,0.023684,0.091270,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",1-5;6-10;11-15;16-20
1,BEIS,current_2026_stronger,0.077565,0.034478,0.077565,0.06,0.15,0.10,0.018066,-0.014153,0.20,0.25,0.080833,0.10,-0.013158,0.027137,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",1-5;6-10;11-15;16-20
2,Travelpro,current_2026_stronger,0.064281,0.023696,0.064281,0.04,0.00,0.00,0.021097,0.001687,0.20,0.40,0.100000,0.15,0.071053,0.046754,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",1-5;6-10;11-15;16-20
3,INUSA,similar_or_small_change,0.019176,0.019722,0.019176,0.06,0.00,0.00,-0.013849,-0.015261,0.40,0.50,0.032798,0.00,-0.007895,-0.015938,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",1-5;6-10;11-15;16-20
4,Delsey,baseline_2023_2024_stronger,-0.079013,-0.091174,-0.079013,-0.10,-0.10,-0.15,-0.048175,-0.046520,-0.65,-0.60,-0.086746,-0.05,-0.034211,-0.057425,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",1-5;6-10;11-15;16-20


## File-check helper

Use this cell if you want to quickly check whether filenames were classified correctly.

- `comparison01_current_included` = used in this notebook.
- `comparison01_ignored` = found but ignored, usually because it is a GEO file or not a current-replacement file.


In [15]:
display(file_manifest_comparison01.sort_values(["included", "file_role", "filename"], ascending=[False, True, True]))


,file_role,path,filename,detected_product,included,reason
0,baseline_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\baseline\baseline_1.txt,baseline_1.txt,,True,baseline answer file
1,baseline_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\baseline\baseline_11.txt,baseline_11.txt,,True,baseline answer file
2,baseline_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\baseline\baseline_16.txt,baseline_16.txt,,True,baseline answer file
3,baseline_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\baseline\baseline_6.txt,baseline_6.txt,,True,baseline answer file
4,comparison01_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\beis_2023vs2026_1.txt,beis_2023vs2026_1.txt,BEIS,True,current replacement file
5,comparison01_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\beis_2023vs2026_11.txt,beis_2023vs2026_11.txt,BEIS,True,current replacement file
6,comparison01_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\beis_2023vs2026_16.txt,beis_2023vs2026_16.txt,BEIS,True,current replacement file
7,comparison01_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\beis_2023vs2026_6.txt,beis_2023vs2026_6.txt,BEIS,True,current replacement file
8,comparison01_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\delsey_2023vs2026_1.txt,delsey_2023vs2026_1.txt,Delsey,True,current replacement file
9,comparison01_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current\delsey_2023vs2026_11.txt,delsey_2023vs2026_11.txt,Delsey,True,current replacement file
